In [1]:
!pip install wandb

In [2]:
import os
import torch
import torch.nn as nn
import numpy as np
import wandb

from torch.utils.data import Dataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shantanugupta2004 (shantanugupta2004-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
N = 128
x = np.linspace(0, 1, N)
y = np.linspace(0, 1, N)

Xg, Yg = np.meshgrid(x, y)
coords_np = np.stack([Xg.flatten(), Yg.flatten()], axis=1)

coords = torch.tensor(coords_np, dtype=torch.float32)

In [4]:
class DeepONetDataset(Dataset):
    def __init__(self, X_data, Y_data, coords, n_points=1000):
        self.X_data = torch.tensor(X_data, dtype=torch.float32)
        self.Y_data = torch.tensor(Y_data, dtype=torch.float32)
        self.coords = coords
        self.n_points = n_points

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):

        branch_input = self.X_data[idx].flatten()

        indices = torch.randint(0, 128*128, (self.n_points,))
        trunk_input = self.coords[indices]

        target_field = self.Y_data[idx].reshape(3, -1).permute(1,0)
        target = target_field[indices]

        return branch_input, trunk_input, target

# Model

In [5]:
class SineLayer(nn.Module):
    def __init__(self, in_features, out_features, omega_0=30, is_first=False):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first

        self.linear = nn.Linear(in_features, out_features)
        self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(
                    -1 / self.linear.in_features,
                    1 / self.linear.in_features
                )
            else:
                self.linear.weight.uniform_(
                    -np.sqrt(6 / self.linear.in_features) / self.omega_0,
                    np.sqrt(6 / self.linear.in_features) / self.omega_0
                )

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

In [6]:
class SirenBranch(nn.Module):
    def __init__(self, input_dim, hidden_dim, p, omega_0=30):
        super().__init__()

        self.net = nn.Sequential(
            SineLayer(input_dim, hidden_dim, omega_0, is_first=True),
            SineLayer(hidden_dim, hidden_dim, omega_0),
            SineLayer(hidden_dim, p, omega_0)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
class SirenTrunk(nn.Module):
    def __init__(self, hidden_dim, p, omega_0=30):
        super().__init__()

        self.net = nn.Sequential(
            SineLayer(2, hidden_dim, omega_0, is_first=True),
            SineLayer(hidden_dim, hidden_dim, omega_0),
            SineLayer(hidden_dim, p, omega_0)
        )

    def forward(self, x):
        return self.net(x)

In [12]:
class SirenDeepONet(nn.Module):
    def __init__(self, branch_dim, hidden_dim=256, p=300, omega_0=30):
        super().__init__()

        self.branch = SirenBranch(branch_dim, hidden_dim, p, omega_0)
        self.trunk = SirenTrunk(hidden_dim, p, omega_0)
        self.output_layer = nn.Linear(p, 3)

    def forward(self, branch_input, trunk_input):

        B, n_pts, _ = trunk_input.shape

        branch_out = self.branch(branch_input)   # [B,p]

        trunk_out = self.trunk(trunk_input.view(-1,2))
        trunk_out = trunk_out.view(B, n_pts, -1)

        combined = branch_out.unsqueeze(1) * trunk_out
        out = self.output_layer(combined)

        return out   # [B,n_pts,3]

In [10]:
def relative_l2(pred, target):
    num = torch.norm(target - pred, dim=(1,2))
    den = torch.norm(target, dim=(1,2))
    return (num / (den + 1e-8)).mean()

# Training

In [13]:
data_path = "/content/drive/MyDrive/LDC Dataset"
geometries = ["harmonics", "nurbs", "skelneton"]

for geometry in geometries:

    wandb.init(
        project="SIREN_DeepONet_LDC_DataOnly",
        name=f"SIREN_{geometry}",
        reinit=True,
        config={
            "epochs": 100,
            "batch_size": 4,
            "lr": 1e-4,   # SIREN prefers smaller LR
            "hidden_dim": 128,
            "p": 100,
            "n_points": 3000,
            "omega_0": 30
        }
    )

    # Load data
    X_data = np.load(os.path.join(
        data_path, f"{geometry}_lid_driven_cavity_X.npz"))["data"]

    Y_data = np.load(os.path.join(
        data_path, f"{geometry}_lid_driven_cavity_Y.npz"))["data"][:,0:3]

    dataset = DeepONetDataset(
        X_data,
        Y_data,
        coords,
        n_points=wandb.config.n_points
    )

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset,
                              batch_size=wandb.config.batch_size,
                              shuffle=True)

    val_loader = DataLoader(val_dataset,
                            batch_size=wandb.config.batch_size)

    model = SirenDeepONet(
        branch_dim=3*128*128,
        hidden_dim=wandb.config.hidden_dim,
        p=wandb.config.p,
        omega_0=wandb.config.omega_0
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=wandb.config.lr)

    mse = nn.MSELoss()

    for epoch in range(wandb.config.epochs):

        # -------- TRAIN --------
        model.train()
        train_loss = 0

        for branch_input, trunk_input, target in train_loader:

            branch_input = branch_input.to(device)
            trunk_input = trunk_input.to(device)
            target = target.to(device)

            optimizer.zero_grad()

            pred = model(branch_input, trunk_input)

            loss = mse(pred, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # -------- VALIDATION --------
        model.eval()
        val_loss = 0
        val_l2 = 0

        with torch.no_grad():
            for branch_input, trunk_input, target in val_loader:

                branch_input = branch_input.to(device)
                trunk_input = trunk_input.to(device)
                target = target.to(device)

                pred = model(branch_input, trunk_input)

                loss = mse(pred, target)

                val_loss += loss.item()
                val_l2 += relative_l2(pred, target).item()

        val_loss /= len(val_loader)
        val_l2 /= len(val_loader)

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_relative_L2": val_l2,
            "learning_rate": optimizer.param_groups[0]["lr"]
        })

        if epoch % 10 == 0:
            print(f"{geometry} | Epoch {epoch} | Val L2: {val_l2:.6f}")

    wandb.finish()

harmonics | Epoch 0 | Val L2: 1.046492
harmonics | Epoch 10 | Val L2: 1.014734
harmonics | Epoch 20 | Val L2: 1.034372
harmonics | Epoch 30 | Val L2: 1.057442
harmonics | Epoch 40 | Val L2: 1.078185
harmonics | Epoch 50 | Val L2: 1.091434


KeyboardInterrupt: 